# IM3 Open Source Data Center Atlas data preparation notebook

In [1]:
# the following libraries are required
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rio
import cartopy as ct
import matplotlib.pyplot as plt
import rasterio as ra
import warnings
from glob import glob

In [ ]:
# Root paths to the raw data

# projected data centers: ask kendall for the latest version of the record if i haven't updated this yet
projected_data_centers_path = './im3_projected_data_centers_v1.1/'

# existing data centers: https://data.msdlive.org/records/65g71-a4731
existing_data_centers_path = './data_center_database'

# fiber density: https://broadbandmap.fcc.gov/home
fiber_provider_density_path = '../bdc_fiber'

# public water service: https://doi.org/10.5066/P9I22Z24
public_water_service_path = '../WSA_v1'

# transmission lines: https://hifld-geoplatform.hub.arcgis.com/datasets/geoplatform::transmission-lines-1/about
transmission_line_path = '../'


### Projected datacenter layers:

In [3]:
scenarios = [
    'high_growth', 'higher_growth', 'low_growth', 'moderate_growth',
]
gravities = [
    0, 25, 50, 75, 100
]

In [4]:
for i in np.arange(len(scenarios)):
    for j in np.arange(len(gravities)):
        d = gpd.read_file(
            f'{projected_data_centers_path}/{scenarios[i]}/{scenarios[i]}_{gravities[j]}_market_gravity.geojson'
        )[[
            'id', 'growth_scenario', 'market_gravity_weight', 'geometry'
        ]]
        d.to_crs('epsg:4326').to_file(
            f'./static/projected/{scenarios[i]}_{gravities[j]}_market_gravity_epsg4326.geojson'
        )
        d['geometry'] = d.centroid
        d.to_crs('epsg:4326').to_file(
            f'./static/projected/{scenarios[i]}_{gravities[j]}_market_gravity_epsg4326_centroids.geojson'
        )

### Fiber provider density layer

In [ ]:
fibs = pd.concat([pd.read_csv(f) for f in sorted(glob(f'{fiber_provider_density_path}/*.csv'))], ignore_index=True)

In [ ]:
fibs = fibs[
    (fibs.max_advertised_download_speed >= 1000) &
    (fibs.max_advertised_upload_speed >= 1000) &
    (fibs.business_residential_code != 'R')
][['h3_res8_id']].groupby('h3_res8_id').size().reset_index().rename(columns={
    'h3_res8_id': 'hexagon',
    0: 'provider_count',
})

In [ ]:
fibs.to_json('./static/fiber_providers_commercial_h3.json', orient='records')

### Public water supply layer

In [ ]:
wat = gpd.read_file(f'{public_water_service_path}/WSA_v1.shp')

In [ ]:
wat[['geometry', 'WSA_NAME']].to_crs('epsg:4326').to_file('../usgs_water_supply.geojson')

In [ ]:
# use tippecanoe to convert the geojson into vector tiles:

In [ ]:
!tippecanoe -e static/tiles/usgs_water_supply --coalesce-densest-as-needed -zg --no-tile-compression ../usgs_water_supply.geojson

In [ ]:
# afterward, update the `static/tiles/usgs_water_supply/metadata.json` such that the values of the keys "center", "bounds", "extent", and "tiles" are proper json numbers or arrays; and also specify the tiles url as /datacenter-atlas/tiles/usgs_water_supply/{z}/{x}/{y}.pbf"

### Transmission line layer

##### If we are using OSM data:

In [ ]:
trans = gpd.read_file(f'{transmission_line_path}/usa_transmission.geojson')

In [ ]:
trans[['voltage', 'geometry']].to_file('../transmission.geojson')

##### If we are using HIFLD data:

In [ ]:
trans = gpd.read_file(f'{transmission_line_path}/transmission_lines.shp.zip')

In [ ]:
trans[['geometry', 'VOLT_CLASS']].to_crs('epsg:4326').to_file('../transmission.geojson')

In [ ]:
# use tippecanoe to convert the geojson into vector tiles:

In [ ]:
!tippecanoe -e static/tiles/transmission --coalesce-densest-as-needed -zg --no-tile-compression ../transmission.geojson

In [ ]:
# afterward, update the `static/tiles/transmission/metadata.json` such that the values of the keys "center", "bounds", "extent", and "tiles" are proper json numbers or arrays; and also specify the tiles url as /datacenter-atlas/tiles/transmission/{z}/{x}/{y}.pbf"

### Existing data centers layer

In [4]:
dcs_points = gpd.read_file(f'{existing_data_centers_path}/im3_us_data_center_locations.gpkg', layer='point')
dcs_buildings = gpd.read_file(f'{existing_data_centers_path}/im3_us_data_center_locations.gpkg', layer='building')
dcs_campuses = gpd.read_file(f'{existing_data_centers_path}/im3_us_data_center_locations.gpkg', layer='campus')

In [ ]:
# we create both a point and a shape layer for these (and another shape layer for campuses)

In [5]:
# we first need to de-duplicate the buildings and points
tmp_points = gpd.sjoin(
    left_df=dcs_points,
    right_df=dcs_buildings[['geometry']],
    how="left"
)
tmp_points = tmp_points[tmp_points.index_right.isna()].drop(columns='index_right')


In [6]:
# select only the campuses that don't also have a building
tmp_campuses = gpd.sjoin(
    left_df=dcs_campuses,
    right_df=dcs_buildings[['geometry']],
    how="left"
)
tmp_campuses = tmp_campuses[tmp_campuses.index_right.isna()].drop(columns='index_right')


In [7]:
# so the unique points are the de-duplicated combination of point, building, and campus layers
unique_dcs = pd.concat([
    tmp_points,
    dcs_buildings,
    tmp_campuses
], ignore_index=True)
unique_dcs['geometry'] = unique_dcs.centroid
unique_dcs[['state_abb', 'county', 'operator', 'name', 'sqft', 'type', 'geometry']].to_file(
    './static/im3_datacenter_centroids.geojson',
    index=False
)

/var/folders/h6/_qg20slx5wn5vthz744t6m5w0000gn/T/ipykernel_60731/4071707056.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  unique_dcs['geometry'] = unique_dcs.centroid


In [8]:
# and the unique polygons are those points with no building (so that they show still at high zoom), plus the buildings and campuses
zoom_dcs = pd.concat([
    tmp_points,
    dcs_campuses,
    dcs_buildings,
], ignore_index=True)
zoom_dcs[['state_abb', 'county', 'operator', 'name', 'sqft', 'type', 'geometry']].to_file(
    './static/im3_datacenter_footprints.geojson',
    index=False
)